# 1.10 — Comprehensions — solutions

Lesson: [`lessons/1.10_comprehensions.md`](../lessons/1.10_comprehensions.md)

Every function body in Problems 1–7 should be a single `return` with a comprehension (or generator expression).

> **Reference solutions.** Open only after you have solved the problem yourself or climbed the hint ladder. Compare, don't copy: then re-solve from blank in a few days.

In [ ]:
# ▶ Run this cell first. It defines check() and expect(), which test your answers,
#   and DATA, the path to the course's shared data folder.
import copy, math
from pathlib import Path

DATA = next((p / "data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "README.md").exists()), Path("data"))

def _same(a, b):
    if hasattr(b, "shape") and hasattr(b, "dtype"):          # numpy arrays
        import numpy as np
        try:
            return np.shape(a) == np.shape(b) and np.allclose(a, b, rtol=1e-5, atol=1e-8)
        except TypeError:
            return False
    if isinstance(b, float) and isinstance(a, (int, float)) and not isinstance(a, bool):
        return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9)
    if isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(b, dict):
        return isinstance(a, dict) and a.keys() == b.keys() and all(_same(a[k], b[k]) for k in b)
    return a == b

def check(fn, cases):
    """Run fn on each (args, expected) pair and report. args is a tuple of arguments."""
    ok = 0
    for args, expected in cases:
        if not isinstance(args, tuple):
            args = (args,)
        shown = ", ".join(repr(a) for a in args)
        if len(shown) > 80:
            shown = shown[:77] + "..."
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ {fn.__name__}({shown}) raised {type(e).__name__}: {e}")
            continue
        if _same(got, expected):
            ok += 1
        else:
            print(f"  ✗ {fn.__name__}({shown}) returned {got!r}, expected {expected!r}")
    status = "✅ all passed" if ok == len(cases) else "keep going"
    print(f"{fn.__name__}: {ok}/{len(cases)} passed — {status}")

def expect(description, condition):
    """For checks that aren't a simple function call: expect('mean is ~0', abs(m) < 1e-6)."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as e:
        print(f"  ✗ {description} — raised {type(e).__name__}: {e}")
        return
    print(("  ✅ " if passed else "  ✗ ") + description + ("" if passed else " — keep going"))


## Problem 1 — squares of evens

In [ ]:
def even_squares(n):
    """Squares of the even numbers from 0 up to (not including) n."""
    return [x ** 2 for x in range(n) if x % 2 == 0]

In [ ]:
check(even_squares, [((10,), [0, 4, 16, 36, 64]), ((1,), [0]), ((0,), [])])

## Problem 2 — filter words

Words longer than 3 letters that start with a vowel, lower-cased.

In [ ]:
def long_vowel_words(words):
    return [w.lower() for w in words if len(w) > 3 and w[0].lower() in "aeiou"]

In [ ]:
check(long_vowel_words, [((["Apple", "ant", "Orange", "kiwi", "eel", "Umbrella"],), ["apple", "orange", "umbrella"])])

## Problem 3 — passing scores (dict comprehension)

In [ ]:
def passing(scores, pass_mark=50):
    return {name: s for name, s in scores.items() if s >= pass_mark}

In [ ]:
check(passing, [(({"a": 49, "b": 50, "c": 90},), {"b": 50, "c": 90}), (({"a": 70}, 80), {})])

## Problem 4 — index records by id

In [ ]:
def index_by_id(records):
    return {r["id"]: r for r in records}

In [ ]:
R = [{"id": 7, "name": "A"}, {"id": 3, "name": "B"}]
check(index_by_id, [((R,), {7: {"id": 7, "name": "A"}, 3: {"id": 3, "name": "B"}})])

## Problem 5 — flatten and transpose

In [ ]:
def flatten(matrix):
    return [x for row in matrix for x in row]

def transpose(matrix):
    return [[row[i] for row in matrix] for i in range(len(matrix[0]))] if matrix else []

In [ ]:
check(flatten, [(([[1, 2], [3], []],), [1, 2, 3])])
check(transpose, [(([[1, 2, 3], [4, 5, 6]],), [[1, 4], [2, 5], [3, 6]])])

## Problem 6 — labels (if/else inside)

Label each amount `"large"` if it's ≥ 10,000, else `"small"`.

In [ ]:
def size_labels(amounts):
    return ["large" if a >= 10_000 else "small" for a in amounts]

In [ ]:
check(size_labels, [(([500, 10_000, 25_000],), ["small", "large", "large"])])

## Problem 7 — totals with generator expressions

In [ ]:
def category_total(transactions, category):
    return sum(t["amount"] for t in transactions if t["category"] == category)

def any_over(transactions, limit):
    return any(t["amount"] > limit for t in transactions)

In [ ]:
T = [{"category": "Dining", "amount": 400.0}, {"category": "Travel", "amount": 9000.0}, {"category": "Dining", "amount": 600.0}]
check(category_total, [((T, "Dining"), 1000.0), ((T, "Health"), 0)])
check(any_over, [((T, 5000), True), ((T, 10_000), False)])

## Problem 8 — read it, then rewrite it as loops

Each function below is written as a comprehension. Write the `_loop` version using plain `for` loops and `if`s — the reading-agent-code skill.

In [ ]:
def mystery_1(words):
    return [w for w in words if len(w) > 3 and w[0] in "aeiou"]

def mystery_2(scores):
    return {k: v * 2 for k, v in scores.items() if v >= 50}

def mystery_3(matrix):
    return [[x * 10 for x in row if x > 0] for row in matrix]

In [ ]:
def mystery_1_loop(words):
    out = []
    for w in words:
        if len(w) > 3 and w[0] in "aeiou":
            out.append(w)
    return out

def mystery_2_loop(scores):
    out = {}
    for k, v in scores.items():
        if v >= 50:
            out[k] = v * 2
    return out

def mystery_3_loop(matrix):
    out = []
    for row in matrix:
        new_row = []
        for x in row:
            if x > 0:
                new_row.append(x * 10)
        out.append(new_row)
    return out

In [ ]:
W = ["apple", "ant", "orange", "kiwi", "idea", "ice"]
S = {"a": 49, "b": 50, "c": 80}
M = [[1, -2, 3], [-1, -1], [4]]
check(mystery_1_loop, [((W,), mystery_1(W))])
check(mystery_2_loop, [((S,), mystery_2(S))])
check(mystery_3_loop, [((M,), mystery_3(M))])

## Problem 9 ★ — word lengths, but only unique words

Return a dict from each unique lower-cased word (strip `.,!?`) to its length, using a single dict comprehension.

In [ ]:
def word_lengths(text):
    return {w.strip(".,!?").lower(): len(w.strip(".,!?")) for w in text.split()}

In [ ]:
check(word_lengths, [(("Data is data. Clean it!",), {"data": 4, "is": 2, "clean": 5, "it": 2})])